# Train PIE on eight datasets

Trains PIE from scratch on each development fold of each dataset, with PIE's within-dataset recipe and all of PIE's own inputs: its five genetic sources, gene text, and cell-type text where it exists.
Training stops early within 2,000 steps and keeps the checkpoint with the best validation AUPRC.

Runs in PIE's own environment ([ArcInstitute/pie](https://github.com/ArcInstitute/pie) at `21a526a`, kernel `pie`, see [Environments](README.md#environments)) on one GPU, one fold at a time, with gradient accumulation 8, the recipe's effective batch of 128.
The Shi runs behind the figure used 2 GPUs with accumulation 4, the same batch.
Each fold's folder, `predictions/pie/<dataset>/fold_<f>/`, holds the split (`split/`), the run (`run/`: checkpoints and a CSV of the metrics) and the test predictions (`pie_f<f>.parquet`).
Rerunning skips finished folds and resumes the others from their last checkpoint.
The `DATASETS` and `FOLDS` environment variables (comma-separated, default all) pick the folds to run, so that one Slurm array task can run one dataset and fold.

In [ ]:
import json
import os
import sys

import numpy as np
import pandas as pd
from pie.data.preprocessed import PreprocessedDir

sys.path.append("../utils")
from paths import data_dir
from pie_train import predict, train

In [ ]:
datasets = (
    pd.read_csv("../0.datasets/datasets.tsv", sep="\t")
    .query("used_in == '2.train_test' and dataset != 'pie_replogle_nadig'")
    .dataset.tolist()
)
datasets = [d for d in datasets if d in os.environ.get("DATASETS", ",".join(datasets)).split(",")]
folds = [int(f) for f in os.environ.get("FOLDS", "0,1,2,3").split(",")]
datasets, folds

PIE's within-dataset recipe (`experiment=replogle_wdataset`: batch 16, learning rate 1e-4, 250 warmup steps, a validation every 100 steps), with two changes made from outside PIE's code, in [`utils/pie_train.py`](../utils/pie_train.py).
Training stops at 2,000 steps, and the learning rate is constant until step 4,500, so a run is the recipe's first 2,000 steps.
Training also stops when validation AUPRC has not improved for 6 validations.

In [ ]:
for dataset in datasets:
    processed = data_dir("perturbseq", "processed", dataset, "preprocessed")  # what 0.datasets/2.process_perturbseq.ipynb's pie prep writes
    pdir = PreprocessedDir.open(processed)
    splits = pd.read_csv(f"../0.datasets/splits/{dataset}.tsv", sep="\t")
    rows = pd.DataFrame(pdir.row_keys(), columns=["context", "symbol"]).merge(splits, on=["context", "symbol"], how="left")
    assert rows.fold.notna().all(), f"{dataset}: a knockdown with no fold"
    for fold in folds:
        out = data_dir("predictions", "pie", dataset, f"fold_{fold}")
        if (out / f"pie_f{fold}.parquet").exists():
            continue

        # test = fold `fold`; validation = 20% of the other development folds' genes (sorted, rng 0); fold 4 is never read
        dev = np.sort(splits.symbol[~splits.fold.isin([fold, 4])].unique())
        val = np.random.default_rng(0).permutation(dev)[: round(0.2 * len(dev))]
        in_dev = ~rows.fold.isin([fold, 4])
        assert set(rows.symbol[in_dev]) == set(dev), f"{dataset}: development genes without a PIE row"
        parts = {"train": in_dev & ~rows.symbol.isin(val), "val": in_dev & rows.symbol.isin(val), "test": rows.fold == fold}
        split = data_dir("predictions", "pie", dataset, f"fold_{fold}", "split")
        for name, part in parts.items():
            keys = {f"{pdir.dataset}.{c}": sorted(rows.symbol[part & (rows.context == c)]) for c in pdir.contexts}
            (split / f"{name}.json").write_text(json.dumps(keys, indent=1) + "\n")
        print(dataset, fold, {name: int(part.sum()) for name, part in parts.items()}, flush=True)

        train(out / "run", processed, split)

        # the best checkpoint predicts the test knockdowns, with the fold's training knockdowns as evidence
        predict(out / "run", split / "test.json", out / f"pie_f{fold}.parquet")